# Module 9A Integrated Lab
## Publications Knowledge Graph — SPARQL over Apache Jena Fuseki

**Domain:** Academic publications (papers, authors, venues, topics).  
**Skills practiced:** Turtle authoring · Fuseki stand-up · 8 SPARQL queries · SKOS disambiguation · GROUP BY · OPTIONAL · CONSTRUCT · pytest integration tests.

---

### Prerequisites

Before running any cell, confirm:

| Check | Command |
|-------|---------|
| Docker Desktop running | Open Docker Desktop |
| Fuseki container up | `docker compose up -d` (in this directory) |
| Dataset loaded | `python load_dataset.py` |
| Python packages installed | `pip install -r requirements.txt` |

---
## Part 0 — Environment Verification

In [1]:
# Cell 0-A: Confirm Python packages are importable
import requests                                        # HTTP library for REST calls
import rdflib                                          # RDF graph parsing and querying
from SPARQLWrapper import JSON, SPARQLWrapper          # SPARQL client and JSON format flag

print(f"rdflib       {rdflib.__version__}")            # Print rdflib version
print(f"requests     {requests.__version__}")          # Print requests version
print("SPARQLWrapper imported OK")                     # Confirm import succeeded

rdflib       7.6.0
requests     2.33.1
SPARQLWrapper imported OK


In [2]:
# Cell 0-B: Ping Fuseki — must return 200 before any query cell will work
resp = requests.get("http://localhost:3030/$/ping", timeout=5)    # Hit Fuseki admin ping endpoint
print(f"Fuseki ping: HTTP {resp.status_code} — {'OK' if resp.status_code == 200 else 'FAILED'}")  # Show pass/fail

Fuseki ping: HTTP 200 — OK


In [3]:
# Cell 0-C: ASK query — verify the publications dataset is loaded
ENDPOINT = "http://localhost:3030/publications/sparql"            # Fuseki SPARQL endpoint URL

sparql = SPARQLWrapper(ENDPOINT)                                  # Create wrapper pointed at endpoint
sparql.setQuery("""
    PREFIX pub: <http://example.org/pub/>
    ASK { ?s a pub:Paper }
""")                                                              # Query: does any Paper triple exist?
sparql.setReturnFormat(JSON)                                      # Request JSON response
result = sparql.query().convert()                                 # Execute query and parse response
print(f"ASK {{ ?s a pub:Paper }} → {result['boolean']}")          # Print true/false result
assert result["boolean"] is True, "Dataset not loaded — run `python load_dataset.py` first"  # Halt if empty

ASK { ?s a pub:Paper } → True


In [4]:
# Cell 0-D: Quick entity counts (sanity check before lab queries)
PREFIXES = """
PREFIX pub:  <http://example.org/pub/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
"""                                                               # Shared prefix block reused by all queries

def _run(query):
    sparql = SPARQLWrapper(ENDPOINT)                              # Fresh wrapper per call
    sparql.setQuery(PREFIXES + query)                             # Prepend shared prefixes
    sparql.setReturnFormat(JSON)                                  # JSON response format
    return sparql.query().convert()["results"]["bindings"]        # Return result rows as a list

counts = {
    "Papers":  _run("SELECT (COUNT(?x) AS ?n) WHERE { ?x a pub:Paper  }")[0]["n"]["value"],    # Count Paper nodes
    "Authors": _run("SELECT (COUNT(?x) AS ?n) WHERE { ?x a pub:Author }")[0]["n"]["value"],    # Count Author nodes
    "Venues":  _run("SELECT (COUNT(?x) AS ?n) WHERE { ?x a pub:Venue  }")[0]["n"]["value"],    # Count Venue nodes
    "Topics":  _run("SELECT (COUNT(?x) AS ?n) WHERE { ?x a skos:Concept }")[0]["n"]["value"],  # Count SKOS Concepts
}
for entity, n in counts.items():
    print(f"  {entity:<10} {n}")                                  # Print each entity type and its count

  Papers     80
  Authors    120
  Venues     40
  Topics     25


---
## Part 1 — SI Demo: Movie Knowledge Graph

The SI instructor walks through three queries on a **movies dataset** so you can see every SPARQL grammar form — SELECT+GROUP BY, OPTIONAL, and CONSTRUCT — before working on the publication suite.

The movie data lives in `movie_demo/movies.ttl`. The queries run locally against **rdflib** (no Fuseki needed for the demo).

### 1.1 Dataset at a Glance

| Entity | Count |
|--------|-------|
| Movies | 17 |
| Directors | 10 |
| Genre concepts (SKOS) | 5 |

Each movie has: `mv:title`, `mv:year`, `mv:genre` (→ SKOS concept), `mv:directedBy` (→ Director), `mv:leadActor` (string literal), and optionally `mv:oscarWin` (boolean).

In [5]:
# Cell 1-A: Load movies.ttl into an rdflib in-memory graph
from pathlib import Path                                          # Cross-platform file path utility

MOVIE_TTL = Path("movie_demo/movies.ttl")                         # Path to the movie Turtle file

g = rdflib.Graph()                                                # Create empty in-memory RDF graph
g.parse(str(MOVIE_TTL), format="turtle")                          # Parse Turtle file into graph
print(f"Loaded {len(g)} triples from {MOVIE_TTL.name}")           # Report how many triples loaded

Loaded 165 triples from movies.ttl


### 1.2 Demo Query 1 — SELECT with GROUP BY

**Question:** How many movies did each director make?

**New grammar forms:**
- `COUNT(?movie)` — aggregate function that counts bindings
- `AS ?movieCount` — renames the aggregate result column
- `GROUP BY ?dirName` — collapses multiple rows per director into one
- `ORDER BY DESC(...)` — sorts largest first

In [6]:
# Cell 1-B: Run Q1 — movies per director
MV_PREFIXES = """
PREFIX mv:   <http://example.org/movie/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
"""                                                               # Namespace prefixes for the movie graph

Q_DEMO1 = MV_PREFIXES + """
SELECT ?dirName (COUNT(?movie) AS ?movieCount) WHERE {
    ?movie     a mv:Movie ;
               mv:directedBy ?dir .
    ?dir       mv:name ?dirName .
}
GROUP BY ?dirName
ORDER BY DESC(?movieCount) ?dirName
"""                                                               # Count movies per director, most first

rows = list(g.query(Q_DEMO1))                                     # Execute query, collect all rows
print(f"{'Director':<28} Movies")
print("-" * 38)                                                   # Print table header and divider
for row in rows:
    print(f"  {str(row.dirName):<26} {row.movieCount}")           # Print each director and their count
print(f"\nTotal director rows: {len(rows)}")                      # Confirm total number of directors

Director                     Movies
--------------------------------------
  Christopher Nolan          3
  Denis Villeneuve           2
  Martin Scorsese            2
  Quentin Tarantino          2
  Stanley Kubrick            2
  Steven Spielberg           2
  David Fincher              1
  David Lynch                1
  Francis Ford Coppola       1
  Wes Anderson               1

Total director rows: 10


### 1.3 Demo Query 2 — SELECT with OPTIONAL

**Question:** Which movies won an Oscar?

**Key insight:** OPTIONAL means the row is *still returned* even if the pattern doesn't match. Movies without an `mv:oscarWin` triple will have `?oscarWin` unbound (Python `None`). Without OPTIONAL, those movies would simply disappear from the results.

In [7]:
# Cell 1-C: Run Q2 — OPTIONAL oscar wins
Q_DEMO2 = MV_PREFIXES + """
SELECT ?title ?year ?oscarWin WHERE {
    ?movie a mv:Movie ;
           mv:title ?title ;
           mv:year  ?year .
    OPTIONAL { ?movie mv:oscarWin ?oscarWin }
}
ORDER BY ?year
"""                                                               # Keeps all movies; Oscar win included if present

rows = list(g.query(Q_DEMO2))                                     # Run query — all 17 movies retained
print(f"{'Title':<48} Year  Oscar")
print("-" * 62)                                                   # Print column headers and divider
for row in rows:
    oscar = str(row.oscarWin) if row.oscarWin is not None else "(unbound)"  # Label missing values
    print(f"  {str(row.title):<46} {row.year}  {oscar}")          # Print title, year, and Oscar status
print(f"\nTotal rows: {len(rows)}  (should equal 17 — all movies retained)")  # Confirm no rows dropped

Title                                            Year  Oscar
--------------------------------------------------------------
  2001: A Space Odyssey                          1968  (unbound)
  The Godfather                                  1972  true
  The Shining                                    1980  (unbound)
  Goodfellas                                     1990  (unbound)
  Schindler's List                               1993  true
  Jurassic Park                                  1993  (unbound)
  Pulp Fiction                                   1994  (unbound)
  Fight Club                                     1999  (unbound)
  Mulholland Drive                               2001  (unbound)
  Inception                                      2010  (unbound)
  Django Unchained                               2012  true
  The Wolf of Wall Street                        2013  (unbound)
  Interstellar                                   2014  (unbound)
  The Grand Budapest Hotel                    

### 1.4 Demo Query 3 — CONSTRUCT

**Question:** Build a new RDF graph where each movie is linked directly to its director's name (not the Director IRI).

**Key insight:** CONSTRUCT returns a *graph* (triples), not a table of rows. You specify a triple template in the `CONSTRUCT { … }` block; the engine instantiates one triple per WHERE-pattern match.

Use case: stripping away the Director node to produce a lighter summary graph for export or visualization.

In [8]:
# Cell 1-D: Run Q3 — CONSTRUCT a directed-by subgraph
Q_DEMO3 = MV_PREFIXES + """
CONSTRUCT {
    ?movie mv:directedByName ?dirName .
}
WHERE {
    ?movie a mv:Movie ;
           mv:directedBy ?dir .
    ?dir   mv:name ?dirName .
}
"""                                                               # Build new graph: movie links directly to director name

result_graph = g.query(Q_DEMO3).graph                             # Execute CONSTRUCT, get returned RDF graph
print(f"Constructed {len(result_graph)} triples.\n")              # Report how many triples were produced
print(f"{'Movie IRI (local name)':<35} directedByName")
print("-" * 65)                                                   # Print column headers and divider
for s, p, o in sorted(result_graph, key=lambda t: str(t[0])):
    local = str(s).split("/")[-1]                                 # Extract local name from the full IRI
    print(f"  {local:<33}  \"{o}\"")                              # Print movie local name and director string

Constructed 17 triples.

Movie IRI (local name)              directedByName
-----------------------------------------------------------------
  arrival                            "Denis Villeneuve"
  bladerunner2049                    "Denis Villeneuve"
  djangounchained                    "Quentin Tarantino"
  dunkirk                            "Christopher Nolan"
  fightclub                          "David Fincher"
  goodfellas                         "Martin Scorsese"
  grandbudapesthotel                 "Wes Anderson"
  inception                          "Christopher Nolan"
  interstellar                       "Christopher Nolan"
  jurassicpark                       "Steven Spielberg"
  mulhollanddrive                    "David Lynch"
  pulpfiction                        "Quentin Tarantino"
  schindlerslist                     "Steven Spielberg"
  thegodfather                       "Francis Ford Coppola"
  theshining                         "Stanley Kubrick"
  thewolfofwallstreet  

---
## Part 2 — Publication Lab (Learner Exercises)

You will now run 8 queries against the **publications** dataset on Fuseki.

Each cell shows a working query. Read the inline comments, run the cell, then answer the corresponding question in `learner_notes.md`.

---

### Q1 — All Papers (SELECT)

The most basic SELECT: retrieve all papers with their title and year.  
**Expected: 80 rows.**

In [9]:
# Cell 2-Q1
Q1 = """
SELECT ?paper ?title ?year WHERE {
    ?paper a pub:Paper ;
           pub:title ?title ;
           pub:year  ?year .
}
ORDER BY ?year ?title
"""                                                               # Fetch all papers sorted by year then title

rows = _run(Q1)                                                   # Execute against Fuseki, get binding list
print(f"Q1 — total papers: {len(rows)}\n")                        # Show total row count
for r in rows[:10]:
    print(f"  {r['year']['value']}  {r['title']['value']}")       # Print year and title for first 10
if len(rows) > 10:
    print(f"  ... ({len(rows) - 10} more rows not shown)")        # Note how many rows were omitted

Q1 — total papers: 80

  2018  Attention Mechanisms for Sequence Transduction
  2018  Bidirectional Pre-training for Language Understanding
  2018  Deep Reinforcement Learning for Robot Navigation
  2018  Graph Convolutional Networks for Entity Classification
  2018  Knowledge Graph Embedding with Relation Projections
  2018  Ontology Alignment Using Semantic Embeddings
  2018  Transfer Learning Across Domains via Domain Adaptation
  2018  Variational Autoencoders with Structured Latent Spaces
  2019  3D Scene Understanding with Point Cloud Networks
  2019  Contextual Embeddings for Aspect Sentiment Analysis
  ... (70 more rows not shown)


---
### Q2 — All Authors (SELECT)

Retrieve every Author entity with their name.  
**Expected: 120 rows.**

In [10]:
# Cell 2-Q2
Q2 = """
SELECT ?author ?name WHERE {
    ?author a pub:Author ;
            pub:name ?name .
}
ORDER BY ?name
"""                                                               # Fetch all authors sorted alphabetically

rows = _run(Q2)                                                   # Run query against Fuseki
print(f"Q2 — total authors: {len(rows)}\n")                       # Show total author count
for r in rows[:8]:
    print(f"  {r['name']['value']}")                              # Print first 8 author names
print(f"  ... ({len(rows) - 8} more)")                            # Note remaining count

Q2 — total authors: 120

  Aarav Mehta
  Abdulrahman Al-Farsi
  Adaeze Obi
  Ahmed Khalil
  Aisha Patel
  Alice Zhang
  Alvaro Mendez
  Amira Mansour
  ... (112 more)


---
### Q3 — Papers per Year (GROUP BY + COUNT)

Aggregate paper counts by year.  
**Expected: 7 rows** (one per year, 2018–2024); all 7 counts must sum to 80.

In [11]:
# Cell 2-Q3
Q3 = """
SELECT ?year (COUNT(?paper) AS ?count) WHERE {
    ?paper a pub:Paper ;
           pub:year ?year .
}
GROUP BY ?year
ORDER BY ?year
"""                                                               # Count papers grouped by publication year

rows = _run(Q3)                                                   # Run aggregation query
print(f"Q3 — year groups: {len(rows)}\n")                         # Print number of distinct years
print(f"  {'Year':<8} Papers")
print("  " + "-" * 18)                                           # Print table header and divider
for r in rows:
    bar = "█" * int(r["count"]["value"])                          # ASCII bar scaled to paper count
    print(f"  {r['year']['value']:<8} {r['count']['value']:<5} {bar}")  # Print year, count, bar chart
total = sum(int(r["count"]["value"]) for r in rows)               # Sum all yearly counts
print(f"\n  Total: {total}")                                       # Verify grand total equals 80

Q3 — year groups: 7

  Year     Papers
  ------------------
  2018     8     ████████
  2019     12    ████████████
  2020     12    ████████████
  2021     16    ████████████████
  2022     14    ██████████████
  2023     12    ████████████
  2024     6     ██████

  Total: 80


---
### Q4 — Prolific Authors (GROUP BY + HAVING)

Find authors who have published **3 or more papers** in the dataset.  
**Expected: 20 rows.**

`HAVING` is the GROUP BY equivalent of `WHERE` — it filters *after* aggregation.

In [12]:
# Cell 2-Q4
Q4 = """
SELECT ?name (COUNT(?paper) AS ?count) WHERE {
    ?paper a pub:Paper ;
           pub:hasAuthor ?author .
    ?author pub:name ?name .
}
GROUP BY ?name
HAVING (COUNT(?paper) >= 3)
ORDER BY DESC(?count) ?name
"""                                                               # Authors with 3+ papers, highest count first

rows = _run(Q4)                                                   # Run HAVING-filtered aggregation
print(f"Q4 — prolific authors (≥3 papers): {len(rows)}\n")        # Show how many authors qualify
for r in rows:
    print(f"  {r['name']['value']:<30} {r['count']['value']} papers")  # Print name and paper count

Q4 — prolific authors (≥3 papers): 20

  Alice Zhang                    3 papers
  Bob Kumar                      3 papers
  Carol Martins                  3 papers
  David Chen                     3 papers
  Eva Kowalski                   3 papers
  Faisal Al-Rashid               3 papers
  Grace Liu                      3 papers
  Hiro Tanaka                    3 papers
  Isabel Ferreira                3 papers
  James O'Brien                  3 papers
  Keiko Nakamura                 3 papers
  Luca Romano                    3 papers
  Maria Santos                   3 papers
  Nikolai Petrov                 3 papers
  Olivia Schmidt                 3 papers
  Pablo Garcia                   3 papers
  Qian Wei                       3 papers
  Rania Hassan                   3 papers
  Stefan Mueller                 3 papers
  Tanya Brown                    3 papers


---
### Q5 — Papers with OPTIONAL DOI

Return all papers, plus their DOI **if one exists**.  
**Expected: 80 rows** — every paper is in the result regardless of whether it has a DOI.

Papers `p001`–`p032` have a DOI; `p033`–`p080` do not. OPTIONAL ensures the DOI-less papers still appear.

In [13]:
# Cell 2-Q5
Q5 = """
SELECT ?title ?year ?doi WHERE {
    ?paper a pub:Paper ;
           pub:title ?title ;
           pub:year  ?year .
    OPTIONAL { ?paper pub:doi ?doi }
}
ORDER BY ?year ?title
"""                                                               # All papers; DOI included only when it exists

rows = _run(Q5)                                                   # Run query; all 80 papers expected
with_doi    = [r for r in rows if "doi" in r]                     # Rows where DOI binding is present
without_doi = [r for r in rows if "doi" not in r]                 # Rows where DOI is absent
print(f"Q5 — total rows : {len(rows)}")                           # Confirm 80 total rows
print(f"     with DOI   : {len(with_doi)}")                       # Should be 32
print(f"     without DOI: {len(without_doi)}")                    # Should be 48
print()
print("Sample (first 5 with DOI):")
for r in with_doi[:5]:
    print(f"  {r['title']['value'][:50]:<52} DOI: {r['doi']['value']}")    # Print title and DOI value
print("\nSample (first 5 without DOI):")
for r in without_doi[:5]:
    print(f"  {r['title']['value'][:50]:<52} DOI: (unbound)")     # Mark missing DOI as unbound

Q5 — total rows : 80
     with DOI   : 32
     without DOI: 48

Sample (first 5 with DOI):
  Attention Mechanisms for Sequence Transduction       DOI: 10.5555/3001.3002
  Bidirectional Pre-training for Language Understand   DOI: 10.5555/3001.3003
  Deep Reinforcement Learning for Robot Navigation     DOI: 10.5555/3002.3003
  Graph Convolutional Networks for Entity Classifica   DOI: 10.5555/3002.3004
  Knowledge Graph Embedding with Relation Projection   DOI: 10.5555/3004.3007

Sample (first 5 without DOI):
  Adaptive Query Rewriting for Semantic Information    DOI: (unbound)
  Argument Mining with Rhetorical Structure Theory     DOI: (unbound)
  Causal Inference with Latent Variable Models         DOI: (unbound)
  Click-Through Rate Prediction with Field-Aware Fac   DOI: (unbound)
  Conversational Recommender Systems with Reinforcem   DOI: (unbound)


---
### Q6 — Semantic Web Papers via SKOS (Property Path)

Find papers whose topic label contains the word **"semantic"** — matching both `skos:prefLabel` and `skos:altLabel` via a property path.

**Expected: 10 rows** (all DISTINCT papers tagged with `pub:topic_sw`).

The SKOS topic `pub:topic_sw` has `prefLabel "Semantic Web"`, so `LCASE(STR(?label))` will contain `"semantic"` for every paper in that topic cluster.

In [14]:
# Cell 2-Q6
Q6 = """
SELECT DISTINCT ?title ?year WHERE {
    ?paper a pub:Paper ;
           pub:title    ?title ;
           pub:year     ?year ;
           pub:hasTopic ?topic .
    ?topic (skos:prefLabel|skos:altLabel) ?label .
    FILTER(CONTAINS(LCASE(STR(?label)), "semantic"))
}
ORDER BY ?year ?title
"""                                                               # Papers whose topic label contains "semantic"

rows = _run(Q6)                                                   # Run property-path + FILTER query
print(f"Q6 — Semantic Web papers: {len(rows)}\n")                 # Print number of matches
for r in rows:
    print(f"  {r['year']['value']}  {r['title']['value']}")       # Print year and title for each match

Q6 — Semantic Web papers: 13

  2018  Knowledge Graph Embedding with Relation Projections
  2018  Ontology Alignment Using Semantic Embeddings
  2020  Crowdsourcing Ontology Construction with Quality Control
  2020  Entity Linking in Heterogeneous Semantic Web Graphs
  2020  Linked Data Integration for Web-Scale Knowledge Graphs
  2020  Ontology Learning from Text with Transformer Encoders
  2020  Reasoning over Linked Open Data with Neural Methods
  2020  SPARQL Query Optimization over Distributed RDF Graphs
  2021  Schema Coverage Analysis for Structured Web Data
  2022  Linked Data Quality Assessment with Machine Learning
  2022  Ontology Modularization Techniques and Applications
  2022  RDF Graph Summarization for Efficient Querying
  2022  SPARQL Federated Queries Across Decentralized Data Pods


---
### Q7 — Recent Papers (FILTER on integer year)

Return all papers published **after 2022** (i.e., 2023 and 2024).  
**Expected: 18 rows** (12 in 2023 + 6 in 2024).

In [15]:
# Cell 2-Q7
Q7 = """
SELECT ?title ?year WHERE {
    ?paper a pub:Paper ;
           pub:title ?title ;
           pub:year  ?year .
    FILTER(?year > 2022)
}
ORDER BY ?year ?title
"""                                                               # Papers published in 2023 or 2024 only

rows = _run(Q7)                                                   # Run FILTER query
print(f"Q7 — papers after 2022: {len(rows)}\n")                   # Expected 18 rows
by_year = {}
for r in rows:
    y = r["year"]["value"]
    by_year.setdefault(y, []).append(r["title"]["value"])          # Group paper titles by year
for year in sorted(by_year):
    print(f"  {year} ({len(by_year[year])} papers)")              # Print year header and count
    for t in by_year[year]:
        print(f"    · {t}")                                        # Print each paper title under its year

Q7 — papers after 2022: 18

  2023 (12 papers)
    · Active Learning Strategies for Annotation Efficiency
    · Biologically Plausible Learning Rules for Deep Networks
    · Cognitive Load Reduction in Intelligent Tutoring Systems
    · Contrastive Learning for Few-Shot Object Detection
    · Human-AI Collaboration Frameworks for Decision Support
    · Neural Radiance Fields for Dynamic Scene Reconstruction
    · Panoramic Depth Estimation with Spherical Transformers
    · Probabilistic Graphical Models for Anomaly Detection
    · Prototype Networks for Class-Incremental Learning
    · Quantum-Enhanced Feature Maps for Kernel Methods
    · Spiking Neural Networks for Energy-Efficient Inference
    · Vision Transformer Fine-Tuning for Dense Prediction
  2024 (6 papers)
    · Association Rule Learning for Drug Interaction Discovery
    · Climate Downscaling with Diffusion Models
    · Flood Forecasting with Spatiotemporal Graph Networks
    · Frequent Pattern Growth for Biomedical Event 

---
### Q8 — Conference Papers Only

Retrieve papers published at **conference** venues (as opposed to journals).  
**Expected: 40 rows** (20 conferences × 2 papers each).

In [16]:
# Cell 2-Q8
Q8 = """
SELECT ?title ?vname WHERE {
    ?paper a pub:Paper ;
           pub:title       ?title ;
           pub:publishedIn ?venue .
    ?venue pub:name ?vname ;
           pub:type "conference" .
}
ORDER BY ?vname ?title
"""                                                               # Papers at venues with type "conference"

rows = _run(Q8)                                                   # Run against Fuseki
print(f"Q8 — conference papers: {len(rows)}\n")                   # Expected 40 rows
by_venue = {}
for r in rows:
    v = r["vname"]["value"]
    by_venue.setdefault(v, []).append(r["title"]["value"])         # Group paper titles by venue name
for venue in sorted(by_venue):
    print(f"  {venue}")                                            # Print venue name as group header
    for t in by_venue[venue]:
        print(f"    · {t}")                                        # Print each paper published there

Q8 — conference papers: 40

  AAAI
    · Knowledge Graph Embedding with Relation Projections
    · Ontology Alignment Using Semantic Embeddings
  ACL
    · Cross-Lingual Transfer for Named Entity Recognition
    · Semantic Role Labeling with Contextual Representations
  CIKM
    · Conversational Recommender Systems with Reinforcement Learning
    · Session-Based Recommendation with Graph Attention Networks
  COLING
    · Argument Mining with Rhetorical Structure Theory
    · Low-Resource Machine Translation with Cross-Lingual Pretraining
  CVPR
    · GAN-Based Data Augmentation for Medical Imaging
    · Saliency Maps for Explainable Neural Networks
  ECCV
    · Adversarial Training for Robust Image Classification
    · Multi-Scale Feature Learning for Video Understanding
  EMNLP
    · Contextual Embeddings for Aspect Sentiment Analysis
    · Question Answering over Knowledge Graphs
  ESWC
    · Entity Linking in Heterogeneous Semantic Web Graphs
    · Ontology Learning from Text with T

---
## Part 3 — rdflib Local Cross-Check

Run Q1 against the local rdflib graph (no server) and compare the row count against Fuseki.  
A mismatch signals a data-load error or a parser difference.

In [17]:
# Cell 3-A: Parse publications.ttl locally
PUB_TTL = Path("data/publications.ttl")                           # Path to the publications Turtle file

local_g = rdflib.Graph()                                          # Create fresh in-memory RDF graph
local_g.parse(str(PUB_TTL), format="turtle")                      # Load and parse the Turtle file
print(f"rdflib loaded {len(local_g)} triples from {PUB_TTL.name}")  # Report triple count loaded

rdflib loaded 1205 triples from publications.ttl


In [18]:
# Cell 3-B: Run Q1 locally and cross-check against Fuseki
Q1_FULL = """
PREFIX pub: <http://example.org/pub/>
SELECT ?paper ?title ?year WHERE {
    ?paper a pub:Paper ;
           pub:title ?title ;
           pub:year  ?year .
}
ORDER BY ?year ?title
"""                                                               # Same Q1 with explicit prefix for local use

local_count  = len(list(local_g.query(Q1_FULL)))                  # Count papers from local rdflib graph
fuseki_count = len(_run(Q1))                                      # Count papers from Fuseki endpoint

print(f"  rdflib  → {local_count} papers")                        # Print local graph result
print(f"  Fuseki  → {fuseki_count} papers")                       # Print Fuseki result

if local_count == fuseki_count:
    print("  PASS — counts match, cross-check successful.")        # Counts agree: data is consistent
else:
    print("  FAIL — count mismatch, check the data load.")         # Mismatch: investigate data load

  rdflib  → 80 papers
  Fuseki  → 80 papers
  PASS — counts match, cross-check successful.


---
## Part 4 — Shutdown

Stop the Fuseki Docker container and release all in-memory RDF graphs.

In [19]:
# Cell 4-A: Shut down Fuseki and release in-memory graphs
import subprocess, pathlib

# Stop and remove the Fuseki Docker container
result = subprocess.run(
    ["docker", "compose", "down"],
    cwd=pathlib.Path(".").resolve(),                               # Run from the lab directory
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("Fuseki container stopped and removed.")
else:
    print("docker compose down output:")
    print(result.stdout or result.stderr)                          # Show output if something went wrong

# Release in-memory RDF graphs
for name, graph in [("publications (local)", local_g), ("movies (demo)", g)]:
    graph.__init__()                                               # Reset graph to empty
    print(f"Cleared {name} graph  ({len(graph)} triples remaining)")

print("\nAll done — Fuseki is down and memory is free.")

Fuseki container stopped and removed.
Cleared publications (local) graph  (0 triples remaining)
Cleared movies (demo) graph  (0 triples remaining)

All done — Fuseki is down and memory is free.
